In [2]:
everything = {}

In [9]:
import os 
import pickle
from vertex_voyage.temporal_partitioning import PartitionerProfile
import numpy as np
cwd = os.getcwd()
from vertex_voyage.partitioning import get_partition_average_balance


def add_to_everything(suffix):
    global everything
    runs_folder = os.path.join(cwd, "..", "temporal_runs", suffix)
    display(f"Searching for profile files in: {runs_folder}")
    for root, dirs, files in os.walk(runs_folder):
        name = os.path.basename(root)
        # name should be in the format "datasetname-partitions_RF"
        name = name.split("-")
        if len(name) < 2:
            # display(f"Skipping {root} as it does not match the expected format.")
            continue
        if len(name[-1].split("_")) < 2:
            # display(f"Skipping {root} as it does not match the expected format.")
            continue
        dataset_name = "-".join(name[:-1])
        parts = name[-1].split("_")[0]
        rf = name[-1].split("_")[1]
        key = (dataset_name, int(parts), int(rf))
        runs = []
        f1s = [] 
        for file in files:
            if file.startswith("profile"):
                file_path = os.path.join(root, file)
                with open(file_path, "rb") as f:
                    run = pickle.load(f)
                    total_vertices = [sum(x // (int(rf)) for x in k.values()) for k in run['partition_sizes']]
                    balance = [get_partition_average_balance(k, len(k.keys())) for k in run['partition_sizes']]
                    run['balance'] = balance
                    run['total_vertices'] = total_vertices
                    run["repartitioning percentage"] = [r / v if v > 0 else 0 for r, v in zip(run['repartitionings'], total_vertices)]
                    if isinstance(run, dict):
                        runs.append(run)
                    else:
                        print(f"File {file_path} is not a PartitionerProfile instance.")
            if file.startswith("iteration_f1s"):
                file_path = os.path.join(root, file)
                with open(file_path, "rb") as f:
                    f1s.append(pickle.load(f))
        if runs:
            average_edge_cuts = np.mean([run['edge_cuts_percentage'] for run in runs], axis=0)
            average_repartitionings = np.mean([run['repartitionings'] for run in runs], axis=0)
            average_repartitioning_percentages = np.mean([run['repartitioning percentage'] for run in runs], axis=0)
            average_balance = np.mean([run['balance'] for run in runs], axis=0)
            average_f1s = np.mean([f1 for f1 in f1s], axis=0) if f1s else None
            stddev_f1s = np.std([f1 for f1 in f1s], axis=0) if f1s else None
            # display(f"Found {len(runs)} profile files in {root}")
            # display(os.path.basename(root))
            # display(f"Average edge cuts: {average_edge_cuts}")
            # display(f"Average repartitionings: {average_repartitionings}")
            # display(f"Average F1 scores: {average_f1s}")
            # display(f"Stddev F1 scores: {stddev_f1s}")
            # display(f"Average repartitioning percentages: {average_repartitioning_percentages}")
            display(f"{dataset_name}: Average F1 score at last iteration: {average_f1s[-1] if average_f1s is not None else 0}")
            # for run in runs:
                # display(run)
            everything[key] = {
                "average_edge_cuts": average_edge_cuts,
                "average_repartitionings": average_repartitionings,
                "average_f1s": average_f1s,
                "stddev_f1s": stddev_f1s,
                "average_repartitioning_percentages": average_repartitioning_percentages,
                "average_balance": average_balance,
                "runs": runs,
                "f1": average_f1s[-1] if average_f1s is not None else 0,
            }
# add_to_everything("hetzner")
# add_to_everything("ai_cluster")
add_to_everything("macbook")
all_datasets = set(key[0] for key in everything.keys())
all_partitions = set(key[1] for key in everything.keys())
all_rfs = set(key[2] for key in everything.keys())


'Searching for profile files in: /Users/stefan/data/Development/VertexVoyage/notebooks/../temporal_runs/macbook'

'DBLP: Average F1 score at last iteration: 0.5434618159411816'

'DBLP: Average F1 score at last iteration: 0.5006599762355777'

'AstroPh: Average F1 score at last iteration: 0.46820128996466837'

'AstroPh: Average F1 score at last iteration: 0.5528067617033892'

...

'AstroPh: Average F1 score at last iteration: 0.5019781231253143'

'DBLP: Average F1 score at last iteration: 0.5429868154452354'

'CITESEER: Average F1 score at last iteration: 0.4700240666630645'

'CITESEER: Average F1 score at last iteration: 0.4545232585871837'

'CITESEER: Average F1 score at last iteration: 0.4986422059691514'

In [10]:
import matplotlib.pyplot as plt
import pandas as pd

for dataset in all_datasets:
    print(f"Dataset: {dataset}")
    data = [] 
    for partitions in all_partitions:
        for rf in [1]:
            key = (dataset, partitions, rf)
            if key in everything:
                f1s = everything[key]["average_f1s"]
                # print("f1s: ", f1s)
                # plt.plot(f1s, label=f"P={partitions}")
                partition_sizes = everything[key]["runs"][0]["partition_sizes"] if everything[key]["runs"] else None
                repartitionings = everything[key]["average_repartitionings"] if everything[key]["average_repartitionings"] is not None else None
                repartitioning_percentages = everything[key]["average_repartitioning_percentages"] if everything[key]["average_repartitioning_percentages"] is not None else None
                edge_cuts = everything[key]["average_edge_cuts"] if everything[key]["average_edge_cuts"] is not None else None
                balance = everything[key]["average_balance"][-1] if everything[key]["average_balance"] is not None else None
                data.append({
                    "# of partitions": partitions,
                    "F1 reconstruction score": f1s[-1] if f1s is not None else 0,
                    "% of edge cuts": everything[key]["average_edge_cuts"][-1] if everything[key]["average_edge_cuts"] is not None else 0,
                    "Balance": balance if balance is not None else 0,
                })
    data = sorted(data, key=lambda x: x["# of partitions"])
    df = pd.DataFrame(data)
    display(df)
    # plt.xlabel("Iteration")
    # plt.ylabel("Average F1 Score")
    # plt.title(f"Average F1 Scores for Dataset {dataset}")
    # plt.legend()
    # plt.show()

Dataset: CITESEER


,# of partitions,F1 reconstruction score,% of edge cuts,Balance
0,1,0.470024,0.000000,1.000000
1,2,0.454523,0.230379,1.308211
2,4,0.474756,0.387125,1.628064
3,8,0.498642,0.480820,1.750000


Dataset: DBLP


,# of partitions,F1 reconstruction score,% of edge cuts,Balance
0,1,0.504444,0.000000,1.000000
1,2,0.543462,0.324796,1.103861
2,4,0.542987,0.504455,1.214044
3,8,0.500660,0.591844,1.346636


Dataset: AstroPh


,# of partitions,F1 reconstruction score,% of edge cuts,Balance
0,1,0.575880,0.000000,1.000000
1,2,0.552807,0.337969,1.102813
2,4,0.501978,0.550119,1.096740
3,8,0.468201,0.646227,1.250000
